# Health-InsurTech : Prediction des frais de sante de maniere ethique

**Mini-Projet Controle Continu**
Dataset : insurance_data.csv — 1 338 clients, 22 variables
Cible : charges (frais medicaux annuels en USD)

---

In [2]:
import os, sys
from pathlib import Path

# Chercher le dossier insurtech_app/ en remontant depuis le repertoire courant
# Fonctionne dans Jupyter local, VS Code, et Google Colab
def find_project_root():
    # Chercher un repere caracteristique : app.py + data/bronze/
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path(__file__).parent if '__file__' in dir() else Path.cwd(),
    ]
    for p in candidates:
        if (p / "data" / "bronze" / "insurance_data.csv").exists():
            return p
    return None

root = find_project_root()
if root:
    os.chdir(root)
    print(f"Repertoire de travail : {root}")
else:
    print(f"Repertoire courant : {os.getcwd()}")
    print("Si data/bronze/insurance_data.csv est introuvable, placez ce notebook")
    print("dans le dossier insurtech_app/ et relancez.")


Repertoire de travail : c:\Users\ACHBARS\Downloads\Health_InsurTech_App\insurtech_app


## Etape 1 : Analyse Ethic-by-Design et RGPD

---

### 1.1 Note d'analyse d'impact — Conformite RGPD

**Contexte**

Dans le cadre du projet Health-InsurTech, nous developpons un outil d'estimation des frais medicaux annuels destine aux futurs clients d'une compagnie d'assurance. Cet outil manipule des donnees a caractere personnel sensibles au sens du Reglement General sur la Protection des Donnees (RGPD, Reglement UE 2016/679), notamment des donnees relatives a la sante (indice de masse corporelle, statut tabagique, antecedents de sante indirects). Il est donc soumis a des obligations strictes de conformite que nous detaillons ci-apres.

**Identification des donnees a caractere personnel et sensibles**

Le dataset contient deux categories de donnees :

Les donnees directement identifiantes sont les suivantes : nom, prenom, date de naissance, adresse email, numero de telephone, numero de securite sociale, adresse postale, code postal, ville, adresse IP et identifiant client. Ces donnees permettent d'identifier directement une personne physique et doivent faire l'objet d'une protection renforcee.

Les donnees indirectement sensibles comprennent : l'age, le sexe, l'indice de masse corporelle (IMC/BMI), le nombre d'enfants, le statut fumeur et les charges medicales. Bien que ne relevant pas formellement de la categorie des "donnees de sante" au sens strict de l'article 9 du RGPD, ces variables permettent d'inferer des informations sur l'etat de sante d'une personne et doivent etre traitees avec precaution.

**Mesures de conformite mises en oeuvre**

Premiere mesure — Anonymisation et pseudonymisation des donnees. Avant toute utilisation dans le modele de Machine Learning, les variables directement identifiantes (nom, prenom, email, telephone, numero de securite sociale, adresse IP) sont supprimees du jeu d'entrainement. Seules les variables strictement necessaires a la prediction sont conservees : age, sexe, IMC, nombre d'enfants, statut fumeur et region. Cette approche respecte le principe de minimisation des donnees pose par l'article 5(1)(c) du RGPD.

Deuxieme mesure — Base legale et consentement eclaire. Le dataset contient une colonne "consentement_rgpd" indiquant que chaque client a explicitement consenti au traitement de ses donnees. Conformement a l'article 6 du RGPD, ce consentement constitue la base legale du traitement. L'application web affichera une notice de consentement claire a l'entree, expliquant quelles donnees sont collectees, dans quel but, pour quelle duree de conservation et avec quels droits pour l'utilisateur (acces, rectification, suppression, portabilite).

Troisieme mesure — Principe de finalite et limitation de la conservation. Les donnees sont utilisees exclusivement aux fins de simulation tarifaire. Elles ne sont pas croisees avec d'autres bases de donnees, ni cedees a des tiers. Aucune donnee saisie par l'utilisateur dans le formulaire de simulation n'est stockee de maniere permanente. Les logs applicatifs, necessaires a la securite, ne contiennent aucune donnee de sante et sont conserves pour une duree maximale de 90 jours.

Quatrieme mesure — Droit a l'explication algorithmique. En vertu de l'article 22 du RGPD, toute decision automatisee ayant un impact significatif sur une personne doit pouvoir etre expliquee. Nous avons deliberement choisi des modeles interpretables (Regression Lineaire, Arbre de Decision) dont les coefficients et les regles de decision sont lisibles et communicables a l'utilisateur. L'interface affiche une explication simplifiee des facteurs ayant le plus influence la prediction.

Cinquieme mesure — Securite des donnees. Conformement a l'article 32 du RGPD, des mesures techniques sont mises en place : communication chiffree via HTTPS, authentification requise pour acceder a l'application, gestion des secrets via Streamlit Secrets (aucune credentiel en clair dans le code), et journalisation des acces.

**Conclusion**

Cette analyse d'impact montre que le projet respecte les grands principes du RGPD : liceite, loyaute, transparence, minimisation des donnees, exactitude, limitation de la conservation, integrite et confidentialite. Une analyse d'impact relative a la protection des donnees (AIPD/DPIA) complete devrait etre conduite avant tout deploiement en production, conformement a l'article 35 du RGPD, en raison du traitement a grande echelle de donnees de sante indirectes.

---

### 1.2 Accessibilite — Conformite RGAA / WCAG 2.1

Le Referentiel General d'Amelioration de l'Accessibilite (RGAA) et les Web Content Accessibility Guidelines (WCAG 2.1) definissent des criteres pour rendre les interfaces web accessibles aux personnes en situation de handicap. Nous implementons les trois mesures concretes suivantes.

**Mesure 1 — Contraste des couleurs suffisant (WCAG 1.4.3, niveau AA)**

Toute combinaison texte/fond respecte un ratio de contraste minimal de 4,5:1 pour le texte courant et de 3:1 pour les grands textes (superieur a 18pt ou 14pt en gras). Dans l'interface Streamlit, cela se traduit par l'utilisation de la palette de couleurs definie dans le fichier .streamlit/config.toml, avec un texte fonce (#1a1a1a) sur fond clair (#ffffff ou #f5f7fa). Les graphiques Plotly utilisent des palettes accessibles aux personnes daltoniennes (palette "Viridis" ou "RdBu" qui evitent les combinaisons rouge/vert problematiques).

**Mesure 2 — Etiquettes et descriptions explicites sur tous les champs de formulaire (WCAG 1.3.1 et 4.1.2)**

Chaque champ de saisie du formulaire de simulation (age, IMC, nombre d'enfants, statut fumeur, sexe, region) est accompagne d'un label explicite, d'une valeur par defaut clairement indiquee et d'un texte d'aide (parametre help des widgets Streamlit). Les messages d'erreur de validation des entrees sont formules de maniere descriptive, indiquant precisement la nature du probleme et la correction attendue. Les sliders affichent la valeur numerique courante en permanence.

**Mesure 3 — Navigation et structure semantique claire (WCAG 1.3.1 et 2.4.6)**

L'application utilise une hierarchie de titres coherente (H1 pour le titre principal de chaque page, H2 pour les sections, H3 pour les sous-sections) respectee dans les cellules Markdown. Le menu de navigation de Streamlit est visible en permanence. Chaque graphique est accompagne d'un titre descriptif et d'un commentaire textuel synthetisant les conclusions principales, de sorte que l'information reste accessible meme sans interpretation visuelle du graphique.

---

## Etape 2 : Modelisation Interpretable

---

### 2.1 Chargement et preparation des donnees

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor, export_text, plot_tree
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11
sns.set_style("whitegrid")

print("Librairies chargees avec succes.")

Librairies chargees avec succes.


In [4]:
# Chargement du dataset
# Le notebook est dans insurtech_app/ — le fichier bronze est dans data/bronze/
df_raw = pd.read_csv("data/bronze/insurance_data.csv")
print(f"Dimensions : {df_raw.shape[0]} lignes x {df_raw.shape[1]} colonnes")
df_raw.head(3)

Dimensions : 1338 lignes x 22 colonnes


,id_client,nom,prenom,date_naissance,sexe,email,telephone,numero_secu_sociale,ville,code_postal,...,sex,bmi,children,smoker,region,charges,mutuelle_complementaire,adresse_ip,consentement_rgpd,date_inscription
0,CLI00001,Laurent,Cécile,04/11/2007,F,cecile.laurent@hotmail.fr,643842498,207116416078825,Bayonne,64100,...,female,27.90,0,yes,southwest,16884.9240,Maaf Santé,245.164.69.170,Oui,19/11/2018
1,CLI00002,Moulin,Guillaume,24/01/2008,M,guillaume_moulin46@gmail.com,682992299,108010691680375,Nice,6000,...,male,33.77,1,no,southeast,1725.5523,Groupama,57.173.29.88,Oui,12/11/2021
2,CLI00003,Joly,Frédéric,08/05/1998,M,fredericjoly@orange.fr,790010943,198050622504036,Antibes,6600,...,male,33.00,3,no,southeast,4449.4620,Aucune,227.165.135.149,Oui,19/06/2024


#### Anonymisation — suppression des variables directement identifiantes

Conformement au principe RGPD de minimisation des donnees, nous supprimons toutes les colonnes
qui permettent d'identifier directement un individu. Seules les variables utiles a la prediction sont conservees.

In [ ]:
# Variables a supprimer (identification directe)
COLS_PII = [
    'id_client', 'nom', 'prenom', 'date_naissance', 'sexe',
    'email', 'telephone', 'numero_secu_sociale',
    'ville', 'code_postal', 'region_fr',
    'adresse_ip', 'consentement_rgpd', 'date_inscription',
    'mutuelle_complementaire'
]

df = df_raw.drop(columns=COLS_PII)

print("Variables conservees pour la modelisation :")
print(df.columns.tolist())
print(f"\nDimensions apres anonymisation : {df.shape}")
df.head()

#### Sauvegarde des donnees Silver anonymisees

In [ ]:
import os

# Creation des dossiers si inexistants
os.makedirs("data/silver", exist_ok=True)

# Silver 1 : donnees anonymisees (variables textuelles conservees)
df.to_csv("data/silver/insurance_anonymized.csv", index=False)
print("Sauvegarde : data/silver/insurance_anonymized.csv")
print(f"  Shape : {df.shape}")

#### Exploration rapide du jeu de donnees anonymise

In [ ]:
print("=== Statistiques descriptives ===")
display(df.describe())

print("\n=== Valeurs manquantes ===")
print(df.isnull().sum())

print("\n=== Distribution des variables categoriques ===")
for col in ['sex', 'smoker', 'region']:
    print(f"  {col} : {df[col].value_counts().to_dict()}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['charges'], bins=40, color='steelblue', edgecolor='white')
axes[0].set_title("Distribution des frais medicaux (charges)")
axes[0].set_xlabel("Charges (USD)")
axes[0].set_ylabel("Nombre de clients")
axes[0].axvline(df['charges'].mean(), color='red', linestyle='--',
                label=f"Moyenne : {df['charges'].mean():,.0f}")
axes[0].axvline(df['charges'].median(), color='orange', linestyle='--',
                label=f"Mediane : {df['charges'].median():,.0f}")
axes[0].legend()

df.boxplot(column='charges', by='smoker', ax=axes[1],
           boxprops=dict(color='steelblue'),
           medianprops=dict(color='red', linewidth=2))
axes[1].set_title("Frais medicaux selon le statut fumeur")
axes[1].set_xlabel("Fumeur (yes/no)")
axes[1].set_ylabel("Charges (USD)")
plt.suptitle("")
plt.tight_layout()
plt.show()

print(f"Charges moyennes — Fumeurs    : {df[df['smoker']=='yes']['charges'].mean():,.0f} USD")
print(f"Charges moyennes — Non-fumeurs: {df[df['smoker']=='no']['charges'].mean():,.0f} USD")
print(f"Ratio fumeurs/non-fumeurs     : {df[df['smoker']=='yes']['charges'].mean() / df[df['smoker']=='no']['charges'].mean():.1f}x")

### 2.2 Encodage et preparation des features

In [ ]:
df_model = df.copy()

ENCODERS = {}
for col in ['sex', 'smoker', 'region']:
    le = LabelEncoder()
    df_model[col + '_enc'] = le.fit_transform(df_model[col])
    ENCODERS[col] = {cls: int(le.transform([cls])[0]) for cls in le.classes_}
    print(f"  {col} : {ENCODERS[col]}")

df_model_clean = df_model.drop(columns=['sex', 'smoker', 'region'])

FEATURES = ['age', 'sex_enc', 'bmi', 'children', 'smoker_enc', 'region_enc']
TARGET   = 'charges'

X = df_model_clean[FEATURES]
y = df_model_clean[TARGET]

print(f"\nFeatures : {FEATURES}")
print(f"Shape X  : {X.shape}  |  Shape y : {y.shape}")

#### Sauvegarde des donnees Silver encodees

In [ ]:
# Silver 2 : donnees encodees pret-a-modeliser
df_model_clean.to_csv("data/silver/insurance_encoded.csv", index=False)
print("Sauvegarde : data/silver/insurance_encoded.csv")
print(f"  Shape : {df_model_clean.shape}")
df_model_clean.head(3)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Entrainement : {X_train.shape[0]} exemples")
print(f"Test         : {X_test.shape[0]} exemples")

### 2.3 Modele 1 — Regression Lineaire

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

mae_lr  = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr   = r2_score(y_test, y_pred_lr)

print("=== Performances — Regression Lineaire ===")
print(f"  MAE  : {mae_lr:,.2f} USD")
print(f"  RMSE : {rmse_lr:,.2f} USD")
print(f"  R2   : {r2_lr:.4f}")

In [ ]:
coef_df = pd.DataFrame({
    'Variable':    FEATURES,
    'Coefficient': lr.coef_
}).sort_values('Coefficient', ascending=False)

print(f"Intercept : {lr.intercept_:,.2f} USD")
display(coef_df)

print(f"\n  Etre fumeur augmente les charges de {coef_df[coef_df['Variable']=='smoker_enc']['Coefficient'].values[0]:,.0f} USD")
print(f"  Chaque annee d'age ajoute {coef_df[coef_df['Variable']=='age']['Coefficient'].values[0]:,.0f} USD")
print(f"  Chaque point d'IMC ajoute {coef_df[coef_df['Variable']=='bmi']['Coefficient'].values[0]:,.0f} USD")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#e74c3c' if c > 0 else '#3498db' for c in coef_df['Coefficient']]
bars = ax.barh(coef_df['Variable'], coef_df['Coefficient'], color=colors, edgecolor='white')
ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_title("Coefficients de la Regression Lineaire (impact en USD)")
ax.set_xlabel("Valeur du coefficient (USD par unite)")
for bar, val in zip(bars, coef_df['Coefficient']):
    ax.text(val + (300 if val >= 0 else -300), bar.get_y() + bar.get_height()/2,
            f"{val:,.0f}", va='center', ha='left' if val >= 0 else 'right', fontsize=9)
red_patch  = mpatches.Patch(color='#e74c3c', label='Augmente les charges')
blue_patch = mpatches.Patch(color='#3498db', label='Diminue les charges')
ax.legend(handles=[red_patch, blue_patch])
plt.tight_layout()
plt.show()

### 2.4 Modele 2 — Arbre de Decision

In [ ]:
dt = DecisionTreeRegressor(max_depth=4, min_samples_leaf=20, random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

mae_dt  = mean_absolute_error(y_test, y_pred_dt)
rmse_dt = np.sqrt(mean_squared_error(y_test, y_pred_dt))
r2_dt   = r2_score(y_test, y_pred_dt)

print("=== Performances — Arbre de Decision (max_depth=4) ===")
print(f"  MAE  : {mae_dt:,.2f} USD")
print(f"  RMSE : {rmse_dt:,.2f} USD")
print(f"  R2   : {r2_dt:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(dt, feature_names=FEATURES, filled=True, rounded=True,
          fontsize=9, ax=ax, impurity=False, precision=0)
ax.set_title("Structure de l'Arbre de Decision (profondeur max = 4)", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
imp_df = pd.DataFrame({
    'Feature':    FEATURES,
    'Importance': dt.feature_importances_
}).sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(imp_df['Feature'], imp_df['Importance'], color='steelblue', edgecolor='white')
ax.set_title("Importance des features — Arbre de Decision")
ax.set_xlabel("Importance relative")
plt.tight_layout()
plt.show()

### 2.5 Modele 3 — Ridge (regularisation)

In [ ]:
ridge = Ridge(alpha=10.0)
ridge.fit(X_train, y_train)
y_pred_ridge = ridge.predict(X_test)

mae_ridge  = mean_absolute_error(y_test, y_pred_ridge)
rmse_ridge = np.sqrt(mean_squared_error(y_test, y_pred_ridge))
r2_ridge   = r2_score(y_test, y_pred_ridge)

print("=== Performances — Ridge (alpha=10) ===")
print(f"  MAE  : {mae_ridge:,.2f} USD")
print(f"  RMSE : {rmse_ridge:,.2f} USD")
print(f"  R2   : {r2_ridge:.4f}")

### 2.6 Comparaison des trois modeles

In [ ]:
comparison = pd.DataFrame({
    'Modele': ['Regression Lineaire', 'Arbre de Decision', 'Ridge'],
    'MAE (USD)':  [mae_lr,    mae_dt,    mae_ridge],
    'RMSE (USD)': [rmse_lr,   rmse_dt,   rmse_ridge],
    'R2':         [r2_lr,     r2_dt,     r2_ridge]
}).round(2)
display(comparison)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
preds  = [y_pred_lr, y_pred_dt, y_pred_ridge]
names  = ['Regression Lineaire', 'Arbre de Decision', 'Ridge']
colors = ['steelblue', 'seagreen', 'darkorange']

for ax, pred, name, col, r2 in zip(axes, preds, names, colors,
                                    [r2_lr, r2_dt, r2_ridge]):
    ax.scatter(y_test, pred, alpha=0.4, color=col, s=18)
    lim = [y_test.min(), y_test.max()]
    ax.plot(lim, lim, 'r--', linewidth=1.5)
    ax.set_title(f"{name}\nR2 = {r2:.3f}")
    ax.set_xlabel("Valeurs reelles (USD)")
    ax.set_ylabel("Valeurs predites (USD)")

plt.tight_layout()
plt.show()

### 2.7 Analyse des biais

In [ ]:
test_idx = X_test.index
df_test  = df.loc[test_idx].copy()
df_test['y_reel']      = y_test.values
df_test['y_predit_lr'] = y_pred_lr
df_test['erreur_abs']  = np.abs(df_test['y_reel'] - df_test['y_predit_lr'])
df_test['erreur_rel']  = df_test['erreur_abs'] / df_test['y_reel'] * 100

mae_global = df_test['erreur_abs'].mean()
print(f"MAE globale : {mae_global:,.2f} USD\n")

for col in ['smoker', 'region', 'sex']:
    print(f"=== Par '{col}' ===")
    print(df_test.groupby(col).agg(
        N=('erreur_abs','count'),
        MAE=('erreur_abs','mean'),
        Erreur_rel=('erreur_rel','mean'),
        Charges_moy=('y_reel','mean')
    ).round(2))
    print()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, col in enumerate(['smoker', 'region', 'sex']):
    groups = df_test[col].unique()
    maes   = [df_test[df_test[col]==g]['erreur_abs'].mean() for g in groups]
    colors = ['#e74c3c' if m > mae_global * 1.1 else '#3498db' for m in maes]
    axes[i].bar(groups, maes, color=colors, edgecolor='white')
    axes[i].axhline(mae_global, color='black', linestyle='--',
                    label=f"MAE globale : {mae_global:,.0f}")
    axes[i].set_title(f"MAE par {col}")
    axes[i].set_ylabel("MAE (USD)")
    axes[i].legend(fontsize=8)

plt.suptitle("Analyse des biais par groupe", fontsize=13)
plt.tight_layout()
plt.show()

### 2.8 Synthese

**Modele retenu : Regression Lineaire**

Ses coefficients sont directement interpretables en USD par unite de variable, ce qui repond
a l'exigence d'explicabilite de l'article 22 du RGPD. Les biais detectes sur le groupe fumeurs
sont signales a l'utilisateur via un intervalle de confiance dans l'application.

---

## Etape 3 : Generation des fichiers pour l'application Streamlit

Cette section genere tous les fichiers necessaires au fonctionnement de l'application :
- Les modeles entraines sauvegardes en `.pkl`
- Les metadonnees (encodeurs, metriques, coefficients)
- Les fichiers de donnees Silver

In [ ]:
import pickle

# Creation du dossier models/ si inexistant
os.makedirs("models", exist_ok=True)

# Sauvegarde des trois modeles
with open("models/linear_regression.pkl", "wb") as f:
    pickle.dump(lr, f)
print("Sauvegarde : models/linear_regression.pkl")

with open("models/decision_tree.pkl", "wb") as f:
    pickle.dump(dt, f)
print("Sauvegarde : models/decision_tree.pkl")

with open("models/ridge.pkl", "wb") as f:
    pickle.dump(ridge, f)
print("Sauvegarde : models/ridge.pkl")

In [ ]:
# Sauvegarde des metadonnees
meta = {
    "features":  FEATURES,
    "target":    TARGET,
    "encoders":  ENCODERS,
    "metrics": [
        {"model": "Regression Lineaire", "MAE": round(mae_lr,2),
         "RMSE": round(rmse_lr,2), "R2": round(r2_lr,4)},
        {"model": "Arbre de Decision",   "MAE": round(mae_dt,2),
         "RMSE": round(rmse_dt,2), "R2": round(r2_dt,4)},
        {"model": "Ridge",               "MAE": round(mae_ridge,2),
         "RMSE": round(rmse_ridge,2), "R2": round(r2_ridge,4)},
    ],
    "coef_lr": {feat: float(coef) for feat, coef in zip(FEATURES, lr.coef_)},
    "intercept_lr": float(lr.intercept_),
    "feature_importance_dt": {feat: float(imp)
                               for feat, imp in zip(FEATURES, dt.feature_importances_)},
    "df_stats": {
        "n_rows":           int(len(df)),
        "smoker_mean":      float(df[df['smoker']=='yes']['charges'].mean()),
        "non_smoker_mean":  float(df[df['smoker']=='no']['charges'].mean()),
        "charges_mean":     float(df['charges'].mean()),
        "charges_median":   float(df['charges'].median()),
        "charges_std":      float(df['charges'].std()),
    }
}

with open("models/meta.pkl", "wb") as f:
    pickle.dump(meta, f)
print("Sauvegarde : models/meta.pkl")

In [ ]:
# Verification finale de tous les fichiers generes
import os

print("=== Fichiers generes ===\n")

for folder, files in [
    ("data/bronze", ["insurance_data.csv"]),
    ("data/silver", ["insurance_anonymized.csv", "insurance_encoded.csv"]),
    ("models",      ["linear_regression.pkl", "decision_tree.pkl",
                     "ridge.pkl", "meta.pkl"]),
]:
    for fname in files:
        path = f"{folder}/{fname}"
        size = os.path.getsize(path) / 1024
        status = "OK" if os.path.exists(path) else "MANQUANT"
        print(f"  [{status}]  {path:<45}  {size:.1f} KB")

print("\nTous les fichiers sont prets pour l'application Streamlit.")

In [ ]:
# Test rapide : rechargement et prediction
with open("models/linear_regression.pkl", "rb") as f:
    lr_loaded = pickle.load(f)

with open("models/meta.pkl", "rb") as f:
    meta_loaded = pickle.load(f)

enc_test = meta_loaded["encoders"]
test_input = pd.DataFrame([{
    "age": 35, "sex_enc": enc_test["sex"]["male"],
    "bmi": 28.0, "children": 0,
    "smoker_enc": enc_test["smoker"]["no"],
    "region_enc": enc_test["region"]["southwest"]
}])

pred_test = float(lr_loaded.predict(test_input)[0])
print(f"Test de rechargement OK")
print(f"Prediction (homme 35 ans, IMC 28, non-fumeur, SW) : {pred_test:,.0f} USD")